# Prepare Frame-Classification Annotation Samples

This notebook creates the human pilot, human validation, and LLM-training pools for the ADHD/autism clinical/lived-experience frame classification stage. It consumes the shared LSC mention table and writes CSV handoffs for annotation.

## Setup

The sampling contract is fixed here so the human and LLM-labelled pools remain disjoint and auditable.

In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_DIR = OUTPUT_DIR / "human_annotation"
LLM_DIR = OUTPUT_DIR / "llm_annotation"

for path in [OUTPUT_DIR, HUMAN_DIR, LLM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260604
PILOT_N = 200
VALIDATION_N = 400
LLM_TRAINING_N = 2000
CODEBOOK_VERSION = "v0.1"
OVERWRITE_EXISTING_ANNOTATION_HANDOFFS = False

## Load Target Contexts

Only ADHD and Autism contexts are frame-labelled. Baseline terms remain unframed comparators.

In [2]:
required_columns = [
    "doc_id",
    "url",
    "registered_domain",
    "analysis_unit",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "target_sentence_plus_adjacent",
    "lsc_year",
    "source_year",
]

contexts = pd.read_parquet(CONTEXT_PATH, columns=required_columns)
target_contexts = contexts.loc[contexts["analysis_unit"].isin(["ADHD", "Autism"])].copy()
target_contexts = target_contexts.dropna(subset=["target_sentence_plus_adjacent", "lsc_year"])
target_contexts = target_contexts.loc[target_contexts["target_sentence_plus_adjacent"].str.strip().ne("")].copy()

def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]

target_contexts["context_id"] = target_contexts.apply(stable_context_id, axis=1)
target_contexts["year_band"] = pd.cut(
    target_contexts["lsc_year"].astype(int),
    bins=[2013, 2017, 2022, 2026],
    labels=["early_2014_2017", "mid_2018_2022", "late_2023_2026"],
)

duplicate_context_ids = target_contexts["context_id"].duplicated().sum()
if duplicate_context_ids:
    raise ValueError(f"Context ID collision or duplicate mention rows found: {duplicate_context_ids}")

target_contexts = target_contexts.sort_values(["analysis_unit", "lsc_year", "context_id"]).reset_index(drop=True)
print(f"Target contexts: {len(target_contexts):,}")
target_contexts.groupby(["analysis_unit", "year_band"], observed=True).size()

Target contexts: 48,403


analysis_unit  year_band      
ADHD           early_2014_2017     5396
               mid_2018_2022       5933
               late_2023_2026      2925
Autism         early_2014_2017    14523
               mid_2018_2022      13359
               late_2023_2026      6267
dtype: int64

## Draw Disjoint Samples

Sampling is stratified by target group and broad year band. The LLM-training pool excludes the human pilot and validation rows.

In [3]:
def stratified_sample(frame: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    group_columns = ["analysis_unit", "year_band"]
    groups = list(frame.groupby(group_columns, observed=True))
    if n > len(frame):
        raise ValueError(f"Requested {n} rows but only {len(frame)} are available")

    base = n // len(groups)
    remainder = n % len(groups)
    sampled_parts = []
    for index, (_, group) in enumerate(groups):
        group_n = min(len(group), base + (1 if index < remainder else 0))
        sampled_parts.append(group.sample(n=group_n, random_state=seed + index))

    sampled = pd.concat(sampled_parts, ignore_index=True)
    shortfall = n - len(sampled)
    if shortfall > 0:
        remaining = frame.loc[~frame["context_id"].isin(sampled["context_id"])]
        sampled = pd.concat([sampled, remaining.sample(n=shortfall, random_state=seed + 999)], ignore_index=True)
    return sampled.sample(frac=1, random_state=seed + 1000).reset_index(drop=True)

pilot = stratified_sample(target_contexts, PILOT_N, RANDOM_SEED)
remaining_after_pilot = target_contexts.loc[~target_contexts["context_id"].isin(pilot["context_id"])].copy()
validation = stratified_sample(remaining_after_pilot, VALIDATION_N, RANDOM_SEED + 10)
remaining_after_human = remaining_after_pilot.loc[~remaining_after_pilot["context_id"].isin(validation["context_id"])].copy()
llm_training = stratified_sample(remaining_after_human, LLM_TRAINING_N, RANDOM_SEED + 20)

assert set(pilot["context_id"]).isdisjoint(validation["context_id"])
assert set(pilot["context_id"]).isdisjoint(llm_training["context_id"])
assert set(validation["context_id"]).isdisjoint(llm_training["context_id"])

for name, sample in [("pilot", pilot), ("validation", validation), ("llm_training", llm_training)]:
    print(name, len(sample))
    print(sample.groupby(["analysis_unit", "year_band"], observed=True).size())

pilot 200
analysis_unit  year_band      
ADHD           early_2014_2017    34
               mid_2018_2022      34
               late_2023_2026     33
Autism         early_2014_2017    33
               mid_2018_2022      33
               late_2023_2026     33
dtype: int64
validation 400
analysis_unit  year_band      
ADHD           early_2014_2017    67
               mid_2018_2022      67
               late_2023_2026     67
Autism         early_2014_2017    67
               mid_2018_2022      66
               late_2023_2026     66
dtype: int64
llm_training 2000
analysis_unit  year_band      
ADHD           early_2014_2017    334
               mid_2018_2022      334
               late_2023_2026     333
Autism         early_2014_2017    333
               mid_2018_2022      333
               late_2023_2026     333
dtype: int64


## Write Annotation Handoffs

Human CSVs include blank label columns. The LLM pool is stored separately for batching in the next notebook.

In [4]:
annotation_columns = [
    "annotation_id",
    "context_id",
    "analysis_unit",
    "lsc_year",
    "raw_form",
    "target_sentence_plus_adjacent",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "clinical_evidence",
    "lived_evidence",
    "uncertainty_note",
    "annotation_round",
    "codebook_version",
]

def human_sheet(sample: pd.DataFrame, prefix: str) -> pd.DataFrame:
    sheet = sample.copy().reset_index(drop=True)
    sheet["annotation_id"] = [f"{prefix}_{i:04d}" for i in range(1, len(sheet) + 1)]
    sheet["clinical_frame_present"] = ""
    sheet["lived_experience_frame_present"] = ""
    sheet["clinical_evidence"] = ""
    sheet["lived_evidence"] = ""
    sheet["uncertainty_note"] = ""
    sheet["annotation_round"] = "pilot" if prefix == "pilot" else "validation"
    sheet["codebook_version"] = CODEBOOK_VERSION
    return sheet[annotation_columns]

pilot_sheet = human_sheet(pilot, "pilot")
validation_sheet = human_sheet(validation, "validation")

pilot_path = HUMAN_DIR / "frame_pilot_annotation_blank.csv"
validation_path = HUMAN_DIR / "frame_validation_annotation_blank.csv"
llm_pool_path = LLM_DIR / "frame_llm_training_pool.csv"
full_pool_path = OUTPUT_DIR / "frame_target_context_pool.csv"

protected_paths = [pilot_path, validation_path, llm_pool_path]
existing_protected_paths = [path for path in protected_paths if path.exists()]
if existing_protected_paths and not OVERWRITE_EXISTING_ANNOTATION_HANDOFFS:
    raise FileExistsError(
        "Annotation handoff(s) already exist. Set OVERWRITE_EXISTING_ANNOTATION_HANDOFFS = True only if you intentionally want to redraw samples: "
        + ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in existing_protected_paths)
    )

pilot_sheet.to_csv(pilot_path, index=False)
validation_sheet.to_csv(validation_path, index=False)
llm_training.to_csv(llm_pool_path, index=False)
target_contexts.to_csv(full_pool_path, index=False)

print("Wrote:")
for path in [pilot_path, validation_path, llm_pool_path, full_pool_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

FileExistsError: Annotation handoff(s) already exist. Set OVERWRITE_EXISTING_ANNOTATION_HANDOFFS = True only if you intentionally want to redraw samples: data/interim/lsc/classification/human_annotation/frame_pilot_annotation_blank.csv, data/interim/lsc/classification/human_annotation/frame_validation_annotation_blank.csv, data/interim/lsc/classification/llm_annotation/frame_llm_training_pool.csv